# Numerical chemistry in practice

Chemistry, understood generally as a change of an species' nature, for whatever reason, is a key component of the wider universe.

Specifically in atmospheres, chemistry determines the presence or absence of absorbers and hence can determine where radiation gets deposited, which again has consequences for atmospheric dynamics. Therefore chemistry really ties together all atmospheric sub-disciplines. It can be crucial to perform their own calculations in specific situations, so in this tutorial we will learn how to let a computer do the most complex of calculations.

This is a teaching resource developed for Master's level physics courses by Matthäus Schulik at Imperial College London.

In [ ]:
from IPython.core.display import display, HTML

display(HTML("<style>.container {width:100% !important;}</style>"))

import numpy as np
import copy
from scipy import *
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as mlines
from scipy.special import lambertw
import scipy.special as sc
from scipy import integrate
matplotlib.get_backend()
from pyhdf.SD import SD
from pyhdf.SD import SDC
from decimal import Decimal
#from h5py.SD import SD

import warnings
warnings.filterwarnings('ignore')

font = {'family': 'serif',
        'color':  'red',
        'weight': 'normal',
        'size': 19,
        }


left1, bottom1, width1, height1 = [0.095, 0.545, 0.88, 0.43]
left2, bottom2, width2, height2 = [0.095, 0.05, 0.88, 0.43]

kb = 1.38e-16
amu= 1.66e-24
G  = 6.678e-8
K_to_eV    = 8.621738e-5;
ev_to_K    = 1./K_to_eV;
Rgas     = 8.31446261815324e7

pi = 3.141592
rpl = 9.45e9
mearth   = 5.98e27
rearth   = 6370e5



# 1.0 Various explicit and implicit solvers for a simple chemical system
## Basic vocabulary

Solving complex problems on computers requires for physical equations to be discretized. The various ways of time-discretization is a conscious choice and has profound consequences on the math needed to solve a system and the numerical instabilities that can occur. Given an ordinary differential equation

\begin{equation}
\frac{\partial y}{\partial t}(x,t) =  f(y,x,t)
\label{eq:hydrogen0} \tag{1} \end{equation}

we discretize time and space with individual steps, denoted by indices $k$ and $j$, i.e. the spatial grid is denoted $x={x^{j}, x^{j+1}, ...}$ and the temporary grid is denoted $t={t^{k}, t^{k+1}, ...}$. Similarly for the variables we solve for, e.g. $y={y^k_j, j^{k+1}_j, ...}$. In the following, we drop the spatial index $j$, as we don't consider spatial gradients in our chemistry problem.

Methods which we call explicit now approximate 

\begin{equation}
\frac{\partial y}{\partial t}(x,t^{k+1}) =  f(y,x,t^k)
\label{eq:hydrogen1} \end{equation}

and are generally easy to code up, as the r.h.s only needs to be known at the current timestep $t^{k}$.
Methods which we call implicit, approximate

\begin{equation}
\frac{\partial y}{\partial t}(x,t^{k+1}) =  f(y,x,t^{k+1})
\label{eq:hydrogen2}\end{equation}

and are much more difficult to code up, as they require knowledge of the r.h.s at the advanced timestep $t^{k+1}$. This is generally not possible to code up, as $f(y,x,t)$ can be a non-linear function. But we can use a popular trick to linearize $f$ via the Taylor-expansion, to be able to separate  $f(y,x,t^{k+1})$ into $f(y,x,t^{k})$ and $y^{t+1}$-components, making this a linear system, which is in principle invertible. Now let's look at this in practice for the dissociation of molecular hydrogen into atomic hydrogen, but we discard atomic hydrogen for now.
I.e. we now take $f= - \alpha\; n_{H2}^2$ and $y=n_{H2}$, so we try to solve the equation


\begin{equation}
\frac{\partial n_{H2}}{\partial t}(x,t^{k+1}) \approx \frac{n_{H2}^{k+1} - n_{H2}^{k}}{\Delta t} =  -\alpha\; n_{H2}^2
\label{eq:hydrogen3}\end{equation}

where $\alpha$ is the reaction coefficient, assumed to be a constant over the timestep $\Delta t$.


## 1.1 The explicit method

Following from the above descriptions, for $H_2 \rightarrow 2 H$ we get

\begin{equation}
\frac{n_{H2}^{k+1} - n_{H2}^{k}}{\Delta t} \approx  -\alpha\; \left( n_{H2}^{k} \right)^2
\label{eq:hydrogen4}\end{equation}

This is trivial to solve for $n_{H2}^{k+1}$ as function of $n_{H2}^{k}$. Code it up, use $\alpha=1$ for now, and check what happens for $\Delta t=10^{-20} s$, $\Delta t=0.1 s$, $\Delta t=10 s$, $\Delta t=10^{20} s$ and try to integrate from $t_{initial}=0s$ to $t_{final}=60 s$.

## 1.2 The implicit method

Now we change the time-index on the r.h.s.: 


\begin{equation}
\frac{n_{H2}^{k+1} - n_{H2}^{k}}{\Delta t} \approx  -\alpha\; \left( n_{H2}^{k+1} \right)^2
\label{eq:hydrogen5}\end{equation}

and suddenly we need to solve a quadratic equation to get to $n_{H2}^{k+1}$. However, when we have done this, we should see why implicit methods are superior over explicit methods, when trying the same timesteps.

## 1.3. The implicit-explicit method

Let us now change the r.h.s according to the Taylor-expansion 

\begin{equation}
f(t^{k+1}) \approx f(t^{k}) + \frac{\partial f(y) }{\partial y}|_{t=t^k} \; \left( y^{k+1} - y^{k}  \right) 
\label{eq:hydrogen6} \end{equation}

the resulting discretized differential equation will be linear and therefore invertible. The form of $n^{k+1} (n^{k})$ will now be different from what we find in 1.1 and 1.2. Comment.

## 1.4 Dissociation-recombination equilibrium

When we consider atomic H, we get a system 

\begin{equation}
\partial_t (H_2) = -\alpha\; (H_2)^2\\
\partial_t (H)   = +2\alpha\; (H_2)^2\\
\end{equation}

and when we consider that atomic H can recombine into molecular H, i.e. we look at the reaction $H_2 \leftrightarrow 2 H$, we need to start counting atomic H as well, and we end up with the system

\begin{equation}
\partial_t (H_2) = -\alpha\; (H_2)^2 + \beta (H)^2\\
\partial_t (H)   = +2\alpha\; (H_2)^2 - 2 \beta (H)^2\\
\end{equation}

where now brackets (X) denote the number density of species X.
To solve the time-evolution of this system, we can employ 1.2 again. Use the conservation of atoms $(H_2) + 2 (H) = const.$ for this.
Our ultimate goal, however, is to solve this system via a matrix method.

## 1.5 Matrix methods

If a system like the above dissociation-recombination system is linearized (e.g. because we discretized it via the implicit-explicit method), then we can invert a matrix to get a solution. 
Write the above nonlinear system as a linearized system in the form of

\begin{equation}
\partial_t \vec{v} \approx \matrix{A}(\vec{v}^{k}) \; \vec{v}^{k+1} + \vec{b}^k
\end{equation}

which will make it possible to write down a solution both analytically, and also via passing the solution matrix to an automated matrix solver. We have now arrived at real applications. 

## 1.6 Numerics homework

Code up the above reaction problems, starting with $H_2 \rightarrow 2H$ and ending with $H_2 \leftrightarrow 2 H$. Explicit methods will guide your way, as you can easily ascertain the correctness of their output (except $\Delta t$ becomes too large). Implicit methods are harder to debug and understand, but stable for all $\Delta t$ and you can ascertain their correctness, when their output is identical to their explicit counterparts.

Matrix methods are the most generally applicable ones for large reaction systems and with reaction terms that have different forms than what was considered above.




# 2.0 Various explicit and implicit solvers for the simplest possible photochemical system

## 2.1 Simple photochemistry as implicit linear system

We are interested in the photoionization of the hydrogen atom, which absorbs photons of energies >$13.6\rm eV$ and subsequently splits into a proton and electron.
The process written as chemical-type reaction is 

\begin{equation}
H+h\nu \rightarrow p^{+} + e^{-}, 
\label{eq:ionisation} \tag{1} \end{equation}

we also consider the reverse reaction, ionic recombination i.e. 

\begin{equation}
H\leftarrow p^{+} + e^{-},
\label{eq:recombination} \tag{2} \end{equation}

where we work under the assumption of case-B recombination, i.e. we assume that the medium in which recombination happens is optically thin to recombination photons (i.e. the optical depth at >13.6eV is <<1), and hence the recombination photon is lost. 
The latter assumption relieves us of the burden of tracking recombination photons in Eqn. $\eqref{eq:recombination}$, at the simple price of using a different recombination coefficient than what would be used for 'true' ionic recombination.
With those assumptions we can formulate the rate equations for the three species involved as
\begin{equation}
\frac{\partial n_H}{\partial t} = - n_H \Gamma + n_e n_p \alpha \\
\frac{\partial n_p}{\partial t} = + n_H \Gamma - n_e n_p \alpha \\
\frac{\partial n_e}{\partial t} = + n_H \Gamma - n_e n_p \alpha \\
\label{eq:system1} \tag{3}
\end{equation}
where we dropped the charge indices. The number densities $n_H$, $n_p$ and $n_e$ in $cm^{-3}$ are the variables to be solved for, $\alpha$ is the case-B recombination coefficient, and

\begin{equation}
\Gamma = \frac{F}{n_H\Delta x}(1-e^{-\Delta \tau}),
\label{eq:iongamma} \tag{4}
\end{equation}

where $F$ is the photon number flux in $cm^{-2}s^{-1}$, $\Delta x$ is the size of the cell in which ionisation is absorbed in $cm$, and the cell optical depth, a dimensionless number, is $\Delta \tau = \Delta x n_H \kappa$ with the hydrogen opacity at that energy $\kappa$ in $cm^{2}/particle$.

Because in this simple system $n_e=n_p$ and $n_e+n_p = const.$ at all times, we can reduce the system of three equations to just one, for the ionization fraction $x\equiv n_e/(n_{\rm tot})$ and $n_{\rm tot} = n_{H}+n_{p}$, which is 

\begin{equation}
\frac{\partial x}{\partial t} =  (1-x)\Gamma - x n_e \alpha \\
                                                  = \frac{n_H F}{n_H \; n_{\rm tot}\Delta x}(1-e^{-d\tau}) - x^2 \alpha n_{\rm tot} \\
                                                  = \frac{F}{n_{\rm tot}\Delta x}(1-e^{-d\tau}) - x^2 \alpha n_{\rm tot}
                                                  \label{eq:ionequation} \tag{5}
\end{equation}

in order to proceed with a numerical method, we will use the semi-implicit Euler method to generate a stable and reliable method. This involves a Taylor-expansion from timestep $k$ to timestep $k+1$ which also denotes variables from now on. Generally, this method approximates 

\begin{equation}
\frac{\partial x}{\partial t} = f(x^{k+1}) \approx f(x^{k}) + \frac{\partial f}{\partial x}|_{x^k} (x^{k+1} - x^{k})
\label{eq:semiimplicit} \tag{6}
\end{equation}

Hence, in practice we need to compute this approximation for both terms. From Eqn. \eqref{eq:ionequation} we read off $f_1=(1-e^{-\Delta\tau})$ and $f_2=x^2$, we further note that $\Delta \tau = \Delta x n_H \kappa = \Delta x \kappa (1-x) n_{tot}$, where we remind of the distinction between the ion fraction $x$ and the numerical cell size $\Delta x$, with which the differentials become

\begin{equation}
\frac{\partial f_1}{\partial x}|_{x^k} = 0-e^{-\Delta\tau}\times \frac{\partial (-\Delta\tau)}{\partial x} |_{x^k} = -e^{-\Delta\tau^k} \Delta x \kappa n_{tot}
\end{equation}

\begin{equation}
\frac{\partial f_2}{\partial x}|_{x^k} = 2x^k
\end{equation}

and with that the full approximation becomes

\begin{equation}
 f_1(x^{k+1}) \approx f_1(x^{k}) + \frac{\partial f_1}{\partial x}|_{x^k} (x^{k+1} - x^{k}) = \frac{F}{n_{tot} \Delta x} \left( (1-e^{-\Delta\tau^k}) -e^{-d\tau^k}\Delta x \kappa n_{tot} (x^{k+1} - x^{k}) \right)
\end{equation}
and
\begin{equation}
 f_2(x^{k+1}) \approx f_2(x^{k}) + \frac{\partial f_2}{\partial x}|_{x^k} (x^{k+1} - x^{k}) = \alpha n_{tot} \left( (x^k)^2  + 2 x^k (x^{k+1} - x^k)   \right)
\end{equation}
we now use $\partial x/\partial t \approx (x^{k+1}-x^{k})/\Delta t$, sort terms with the advanced time $x^{k+1}$ to the l.h.s and all other terms to the r.h.s, resulting in our final implicit numerical scheme

\begin{equation}
    x^{k+1} \left(1 + \Delta t F \kappa e^{-\Delta\tau^k} +  2 \Delta t n_{tot} \alpha x^k \right) = x^k \left( 1 + \Delta t F \kappa e^{-\Delta\tau^k} + \Delta t n_{tot} \alpha x^k \right) + \Delta t\frac{F}{n_{tot} \Delta x}  \left(1-e^{-\Delta\tau^k}\right)
\end{equation}

FYI this solution method gives the same solution as C2-Ray (Mellema+2006). In a completely analoguous way one can derive the implicit system for the non-reduced two or three-equation system, to get a more generalizable formulation in case one wants to add more chemistry processes/reactions.

## 2.2 Numerics homework

The above reaction system can be solved in two ways. The simplest way is to just take the last equation, which has an explicit solution for $x^{k+1}$, and time-evolve that. The other way is to understand the original, but linearized system of equation as linear system, which makes it eligible to be solve via a matrix-method.


In [ ]:
#Space for your own work






# 3.0 Solvers for a simple 4-component equation

## 3.1 Theory

We want to solve the time-evolution for the reaction

$$
\begin{aligned}
CO + 3H_2 \leftrightarrow CH_4 + H_2O,
\end{aligned}
\tag{1}
$$

which has constraints on total number of atoms as follows, and we drop indices from number densities, instead writing $n_X \equiv (X)$,

\begin{equation}
(C) = (CO) + (CH_4) \\
(O) = (CO) + (H_2O) \\
(H) = 2(H_2) + 4(CH_4) + 2(H_2O)  \tag{2}
\end{equation}

The evolution equations resulting from this are

\begin{equation}
\frac{1}{1} \partial_t (CO) = -(CO)(H_2)^3 k_f + (CH_4)(H_2O) k_r\\
\frac{1}{3} \partial_t (H_2) = -(CO)(H_2)^3 k_f + (CH_4)(H_2O) k_r\\
\frac{1}{1} \partial_t (CH_4) = +(CO)(H_2)^3 k_f - (CH_4)(H_2O) k_r\\
\frac{1}{1} \partial_t (H_2O) = +(CO)(H_2)^3 k_f - (CH_4)(H_2O) k_r
\end{equation}

where $k_f$ and $k_r$ are the forward and backward reaction rates. Cooper & Showman(2006) discuss how those evolution equations are actually wrong, because reactive intermediate steps exist in Eqn. \eqref{eq:cochemistry} and hence this reaction has to be treated as a reaction chain, such as discussed in the lectures at https://cefrc.princeton.edu/sites/cefrc/files/Files/2015%20Lecture%20Notes/Wang/Lecture-3-Basic-Chemical-Kinetics.pdf
Nonetheless, we will solve this system with the given, naive, kinetic reaction terms, as numerical exercise.

## 3.2 Numerics homework

We will employ three different methods, 1) the explicit solution of the 4-equation system, 2) the matrix-aided semi-implicit 4-equation system and 3) a semi-implicit solution of the reduced system to one equation, via the help of the atom conservation statements.

In [ ]:
#Space for your own work




